# AWS Marketplace EventBridge Notifications Test Generator

This notebook generates random test messages for all AWS Marketplace EventBridge notification use cases and sends them to EventBridge for testing the capture infrastructure.

## Event Types Covered:
- Purchase Agreement Created (Manufacturer, Proposer, Acceptor)
- Purchase Agreement Amended (Manufacturer, Proposer, Acceptor)
- Purchase Agreement Ended (Manufacturer, Proposer, Acceptor)
- License Updated (Manufacturer)
- License Deprovisioned (Manufacturer)

In [ ]:
# Install required packages
!pip install boto3 faker

In [2]:
import boto3
import json
import uuid
import random
from datetime import datetime, timedelta, timezone
from faker import Faker
import time
from typing import Dict, List, Any

# Initialize Faker for generating realistic test data
fake = Faker()

# create a boto3 session and provide profile and region
session = boto3.Session(
    profile_name=None,
    region_name='us-east-1'
)

# Initialize AWS clients
eventbridge_client = session.client('events')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## Configuration and Test Data Generators

In [3]:
# Configuration
#EVENT_SOURCE = 'aws.agreement-marketplace'
EVENT_SOURCE = 'mock.agreement-marketplace'

REGION = 'us-east-1'
CATALOG = 'AWSMarketplace'

# Test data pools
AGREEMENT_STATUSES = ['ACTIVE', 'CANCELLED', 'EXPIRED', 'TERMINATED', 'RENEWED', 'REPLACED']
AGREEMENT_INTENTS = ['NEW', 'AMEND', 'RENEW', 'REPLACE']
PRODUCT_CODES = [
    '4qmf911dci742labk3zt6i3i7',
    'cqj79vcg9mig83heufcyjpfai',
    '82w69ekz7sauvzjcgxjry8tv0'
]

def generate_aws_account_id() -> str:
    """Generate a realistic AWS account ID"""
    return str(random.randint(100000000000, 999999999999))

def generate_agreement_id() -> str:
    """Generate a realistic agreement ID"""
    return f"agmt-{fake.lexify('?' * 25, letters='abcdefghijklmnopqrstuvwxyz0123456789')}"

def generate_offer_id() -> str:
    """Generate a realistic offer ID"""
    return f"offer-{fake.lexify('?' * 16, letters='abcdefghijklmnopqrstuvwxyz0123456789')}"

def generate_resale_auth_id() -> str:
    """Generate a realistic resale authorization ID"""
    return f"resaleauthz-{fake.lexify('?' * 13, letters='abcdefghijklmnopqrstuvwxyz0123456789')}"

def generate_license_id() -> str:
    """Generate a realistic license ID"""
    return f"l-{fake.lexify('?' * 32, letters='abcdefghijklmnopqrstuvwxyz0123456789')}"

def generate_product_id() -> str:
    """Generate a realistic product ID or use provided one"""
    return f"prod-{fake.lexify('?' * 13, letters='abcdefghijklmnopqrstuvwxyz0123456789')}"

def generate_timestamps():
    """Generate realistic timestamps for agreement lifecycle"""
    now = datetime.now(timezone.utc)
    acceptance_time = now - timedelta(days=random.randint(1, 30))
    start_time = acceptance_time + timedelta(hours=random.randint(1, 24))
    end_time = start_time + timedelta(days=random.randint(30, 365))
    
    return {
        'current': now.isoformat().replace('+00:00', 'Z'),
        'acceptance': acceptance_time.isoformat().replace('+00:00', 'Z'),
        'start': start_time.isoformat().replace('+00:00', 'Z'),
        'end': end_time.isoformat().replace('+00:00', 'Z')
    }

print("✅ Configuration and generators ready")

✅ Configuration and generators ready


## Event Generation Functions

In [4]:
def create_base_event(detail_type: str, account_id: str, agreement_id: str) -> Dict[str, Any]:
    """Create base EventBridge event structure"""
    timestamps = generate_timestamps()
    
    return {
        'version': '0',
        'id': str(uuid.uuid4()),
        'detail-type': detail_type,
        'source': EVENT_SOURCE,
        'account': account_id,
        'time': timestamps['current'],
        'region': REGION,
        'resources': [f'arn:aws:aws-marketplace::aws:agreement:{agreement_id}'],
        'detail': {
            'requestId': str(uuid.uuid4()),
            'catalog': CATALOG,
            'agreementId': agreement_id
        }
    }

print("✅ Base event function ready")

✅ Base event function ready


In [12]:
# Purchase Agreement Created Events
def generate_purchase_agreement_created_manufacturer() -> Dict[str, Any]:
    """Generate Purchase Agreement Created - Manufacturer event"""
    agreement_id = generate_agreement_id()
    manufacturer_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    timestamps = generate_timestamps()
    
    event = create_base_event('Purchase Agreement Created - Manufacturer', manufacturer_account, agreement_id)
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'intent': random.choice(['NEW', 'RENEW', 'REPLACE']),
            'status': 'ACTIVE',
            'acceptanceTime': timestamps['acceptance'],
            'startTime': timestamps['start'],
            'endTime': timestamps['end']
        },
        'resaleAuthorization': {
            'id': generate_resale_auth_id()
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_purchase_agreement_created_proposer() -> Dict[str, Any]:
    """Generate Purchase Agreement Created - Proposer event"""
    agreement_id = generate_agreement_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    timestamps = generate_timestamps()
    
    event = create_base_event('Purchase Agreement Created - Proposer', proposer_account, agreement_id)
    
    # Randomly decide if this is a channel partner offer (has resale auth) or direct offer (null)
    is_channel_partner = random.choice([True, False])
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'intent': random.choice(['NEW', 'RENEW', 'REPLACE']),
            'status': 'ACTIVE',
            'acceptanceTime': timestamps['acceptance'],
            'startTime': timestamps['start'],
            'endTime': timestamps['end']
        },
        'resaleAuthorization': {
            'id': generate_resale_auth_id() if is_channel_partner else None
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_purchase_agreement_created_acceptor() -> Dict[str, Any]:
    """Generate Purchase Agreement Created - Acceptor event"""
    agreement_id = generate_agreement_id()
    acceptor_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    timestamps = generate_timestamps()
    
    event = create_base_event('Purchase Agreement Created - Acceptor', acceptor_account, agreement_id)
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'intent': random.choice(['NEW', 'RENEW', 'REPLACE']),
            'status': 'ACTIVE',
            'acceptanceTime': timestamps['acceptance'],
            'startTime': timestamps['start'],
            'endTime': timestamps['end']
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

print("✅ Purchase Agreement Created events ready")

✅ Purchase Agreement Created events ready


In [11]:
# Purchase Agreement Amended Events
def generate_purchase_agreement_amended_manufacturer() -> Dict[str, Any]:
    """Generate Purchase Agreement Amended - Manufacturer event"""
    agreement_id = generate_agreement_id()
    manufacturer_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    timestamps = generate_timestamps()
    
    event = create_base_event('Purchase Agreement Amended - Manufacturer', manufacturer_account, agreement_id)
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'intent': 'AMEND',
            'status': 'ACTIVE',
            'acceptanceTime': timestamps['acceptance'],
            'startTime': timestamps['start'],
            'endTime': timestamps['end']
        },
        'resaleAuthorization': {
            'id': generate_resale_auth_id()
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_purchase_agreement_amended_proposer() -> Dict[str, Any]:
    """Generate Purchase Agreement Amended - Proposer event"""
    agreement_id = generate_agreement_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    timestamps = generate_timestamps()
    
    event = create_base_event('Purchase Agreement Amended - Proposer', proposer_account, agreement_id)
    
    is_channel_partner = random.choice([True, False])
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'intent': 'AMEND',
            'status': 'ACTIVE',
            'acceptanceTime': timestamps['acceptance'],
            'startTime': timestamps['start'],
            'endTime': timestamps['end']
        },
        'resaleAuthorization': {
            'id': generate_resale_auth_id() if is_channel_partner else None
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_purchase_agreement_amended_acceptor() -> Dict[str, Any]:
    """Generate Purchase Agreement Amended - Acceptor event"""
    agreement_id = generate_agreement_id()
    acceptor_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    timestamps = generate_timestamps()
    
    event = create_base_event('Purchase Agreement Amended - Acceptor', acceptor_account, agreement_id)
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'intent': 'AMEND',
            'status': 'ACTIVE',
            'acceptanceTime': timestamps['acceptance'],
            'startTime': timestamps['start'],
            'endTime': timestamps['end']
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

print("✅ Purchase Agreement Amended events ready")

✅ Purchase Agreement Amended events ready


In [10]:
# Purchase Agreement Ended Events
def generate_purchase_agreement_ended_manufacturer() -> Dict[str, Any]:
    """Generate Purchase Agreement Ended - Manufacturer event"""
    agreement_id = generate_agreement_id()
    manufacturer_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    
    event = create_base_event('Purchase Agreement Ended - Manufacturer', manufacturer_account, agreement_id)
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'status': random.choice(['CANCELLED', 'EXPIRED', 'RENEWED', 'REPLACED', 'TERMINATED'])
        },
        'resaleAuthorization': {
            'id': generate_resale_auth_id()
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_purchase_agreement_ended_proposer() -> Dict[str, Any]:
    """Generate Purchase Agreement Ended - Proposer event"""
    agreement_id = generate_agreement_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    
    event = create_base_event('Purchase Agreement Ended - Proposer', proposer_account, agreement_id)
    
    is_channel_partner = random.choice([True, False])
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'status': random.choice(['CANCELLED', 'EXPIRED', 'RENEWED', 'REPLACED', 'TERMINATED'])
        },
        'resaleAuthorization': {
            'id': generate_resale_auth_id() if is_channel_partner else None
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_purchase_agreement_ended_acceptor() -> Dict[str, Any]:
    """Generate Purchase Agreement Ended - Acceptor event"""
    agreement_id = generate_agreement_id()
    acceptor_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    
    event = create_base_event('Purchase Agreement Ended - Acceptor', acceptor_account, agreement_id)
    
    event['detail'].update({
        'agreement': {
            'id': agreement_id,
            'status': random.choice(['CANCELLED', 'EXPIRED', 'RENEWED', 'REPLACED', 'TERMINATED'])
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

print("✅ Purchase Agreement Ended events ready")

✅ Purchase Agreement Ended events ready


In [9]:
# License Events
def generate_license_updated_manufacturer(product_id: str = None) -> Dict[str, Any]:
    """Generate License Updated - Manufacturer event"""
    agreement_id = generate_agreement_id()
    manufacturer_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    
    event = create_base_event('License Updated - Manufacturer', manufacturer_account, agreement_id)
    
    if not product_id:
        product_id = generate_product_id()

    event['detail'].update({
        'agreement': {
            'id': agreement_id
        },
        'product': {
            'code': random.choice(PRODUCT_CODES),
            'id': product_id
        },
        'license': {
            'id': generate_license_id()
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

def generate_license_deprovisioned_manufacturer(product_id: str = None) -> Dict[str, Any]:
    """Generate License Deprovisioned - Manufacturer event"""
    agreement_id = generate_agreement_id()
    manufacturer_account = generate_aws_account_id()
    proposer_account = generate_aws_account_id()
    acceptor_account = generate_aws_account_id()
    
    event = create_base_event('License Deprovisioned - Manufacturer', manufacturer_account, agreement_id)
    
    if not product_id:
        product_id = generate_product_id()

    event['detail'].update({
        'agreement': {
            'id': agreement_id
        },
        'product': {
            'code': random.choice(PRODUCT_CODES),
            'id': product_id
        },
        'license': {
            'id': generate_license_id()
        },
        'acceptor': {
            'accountId': acceptor_account
        },
        'proposer': {
            'accountId': proposer_account
        },
        'offer': {
            'id': generate_offer_id()
        }
    })
    
    return event

print("✅ License events ready")

✅ License events ready


## Event Registry and Utilities

In [13]:
# Event generator registry
EVENT_GENERATORS = {
    'Purchase Agreement Created - Manufacturer': generate_purchase_agreement_created_manufacturer,
    'Purchase Agreement Created - Proposer': generate_purchase_agreement_created_proposer,
    'Purchase Agreement Created - Acceptor': generate_purchase_agreement_created_acceptor,
    'Purchase Agreement Amended - Manufacturer': generate_purchase_agreement_amended_manufacturer,
    'Purchase Agreement Amended - Proposer': generate_purchase_agreement_amended_proposer,
    'Purchase Agreement Amended - Acceptor': generate_purchase_agreement_amended_acceptor,
    'Purchase Agreement Ended - Manufacturer': generate_purchase_agreement_ended_manufacturer,
    'Purchase Agreement Ended - Proposer': generate_purchase_agreement_ended_proposer,
    'Purchase Agreement Ended - Acceptor': generate_purchase_agreement_ended_acceptor,
    'License Updated - Manufacturer': generate_license_updated_manufacturer,
    'License Deprovisioned - Manufacturer': generate_license_deprovisioned_manufacturer
}

def send_event_to_eventbridge(event: Dict[str, Any]) -> bool:
    """Send event to EventBridge"""
    try:
        response = eventbridge_client.put_events(
            Entries=[
                {
                    'Source': event['source'],
                    'DetailType': event['detail-type'],
                    'Detail': json.dumps(event['detail']),
                    'Resources': event['resources']
                }
            ]
        )
        print(f'response: {json.dumps(response, indent=2, default=str)}')
        if response['FailedEntryCount'] == 0:
            print(f"✅ Successfully sent: {event['detail-type']}")
            return True
        else:
            print(f"❌ Failed to send: {event['detail-type']} - {response['Entries'][0].get('ErrorMessage', 'Unknown error')}")
            return False
            
    except Exception as e:
        print(f"❌ Error sending {event['detail-type']}: {str(e)}")
        return False

def generate_and_send_event(event_type: str) -> Dict[str, Any]:
    """Generate and send a specific event type"""
    if event_type not in EVENT_GENERATORS:
        print(f"❌ Unknown event type: {event_type}")
        return None
    
    event = EVENT_GENERATORS[event_type]()
    success = send_event_to_eventbridge(event)
    
    return event if success else None

def generate_and_send_all_events() -> List[Dict[str, Any]]:
    """Generate and send one of each event type"""
    sent_events = []
    
    for event_type in EVENT_GENERATORS.keys():
        print(f"\n🔄 Generating {event_type}...")
        event = generate_and_send_event(event_type)
        if event:
            sent_events.append(event)
        time.sleep(1)  # Small delay between events
    
    return sent_events

print("✅ Event utilities ready")
print(f"📊 Available event types: {len(EVENT_GENERATORS)}")
for event_type in EVENT_GENERATORS.keys():
    print(f"  - {event_type}")

✅ Event utilities ready
📊 Available event types: 11
  - Purchase Agreement Created - Manufacturer
  - Purchase Agreement Created - Proposer
  - Purchase Agreement Created - Acceptor
  - Purchase Agreement Amended - Manufacturer
  - Purchase Agreement Amended - Proposer
  - Purchase Agreement Amended - Acceptor
  - Purchase Agreement Ended - Manufacturer
  - Purchase Agreement Ended - Proposer
  - Purchase Agreement Ended - Acceptor
  - License Updated - Manufacturer
  - License Deprovisioned - Manufacturer


## Test Execution

In [12]:
# Generate and preview a sample event without sending
sample_event = generate_purchase_agreement_created_manufacturer()
print("📋 Sample Event Preview:")
print(json.dumps(sample_event, indent=2))

📋 Sample Event Preview:
{
  "version": "0",
  "id": "3e31c215-1dde-4d96-89d3-742de586a33a",
  "detail-type": "Purchase Agreement Created - Manufacturer",
  "source": "mock.agreement-marketplace",
  "account": "751425604044",
  "time": "2025-09-10T12:12:05.372871Z",
  "region": "us-east-1",
  "resources": [
    "arn:aws:aws-marketplace::aws:agreement:agmt-3ly0j6e0khsnfu9sqgm3j3hn7"
  ],
  "detail": {
    "requestId": "d7405941-95f0-456e-b9ae-cc3afed81986",
    "catalog": "AWSMarketplace",
    "agreementId": "agmt-3ly0j6e0khsnfu9sqgm3j3hn7",
    "agreement": {
      "id": "agmt-3ly0j6e0khsnfu9sqgm3j3hn7",
      "intent": "NEW",
      "status": "ACTIVE",
      "acceptanceTime": "2025-08-18T12:12:05.372842Z",
      "startTime": "2025-08-18T20:12:05.372842Z",
      "endTime": "2026-02-18T20:12:05.372842Z"
    },
    "resaleAuthorization": {
      "id": "resaleauthz-qjyqn36b7dxua"
    },
    "acceptor": {
      "accountId": "903412889104"
    },
    "proposer": {
      "accountId": "50502820

In [ ]:
# Send a single test event
print("🚀 Sending test events...")
test_events = []
test_events.append(generate_and_send_event('Purchase Agreement Created - Manufacturer'))
time.sleep(2)
#test_event = generate_and_send_event('Purchase Agreement Created - Proposer')
#test_event = generate_and_send_event('Purchase Agreement Amended - Proposer')

test_events.append(generate_and_send_event('License Updated - Manufacturer'))

#print(json.dumps(test_event, indent=2, default=str))

if test_events:
    print("\n\n")
    print('#' * 60)
    for test_event in test_events:
        print(f"✅ Test event sent successfully!")
        print(f"Detail Type: {test_event['detail-type']}")
        print(f"Agreement ID: {test_event['detail']['agreementId']}")
        print(f"Event ID: {test_event['id']}")
        print('#' * 60)

## License - entitlement events

Send license events to **EventBridge**.

### Event with random product id

In [ ]:
event = generate_license_updated_manufacturer()
print(json.dumps(event, indent=2, default=str))
send_event_to_eventbridge(event)

### Event with given product id

The product id must match the product id in the **EventBridge** rule.

In [18]:
product_id = 'prod-2zch4ci4slnp2' # product id in EventBridge rule
event = generate_license_updated_manufacturer(product_id)
print(json.dumps(event, indent=2, default=str))
send_event_to_eventbridge(event)

{
  "version": "0",
  "id": "f38bc7b3-dd36-4107-afae-cbf2e41b598c",
  "detail-type": "License Updated - Manufacturer",
  "source": "mock.agreement-marketplace",
  "account": "817077747771",
  "time": "2025-09-08T15:37:07.410890Z",
  "region": "us-east-1",
  "resources": [
    "arn:aws:aws-marketplace::aws:agreement:agmt-j31sgfmgq88f8y85kqpsq0j9r"
  ],
  "detail": {
    "requestId": "b678e33a-c297-4f2c-a0cc-4e2e0abf03b0",
    "catalog": "AWSMarketplace",
    "agreementId": "agmt-j31sgfmgq88f8y85kqpsq0j9r",
    "agreement": {
      "id": "agmt-j31sgfmgq88f8y85kqpsq0j9r"
    },
    "product": {
      "code": "4qmf911dci742labk3zt6i3i7",
      "id": "prod-2zch4ci4slnp2"
    },
    "license": {
      "id": "l-cliylbbgx0armw2bd83kqj12fyhatc6y"
    },
    "acceptor": {
      "accountId": "386582279003"
    },
    "proposer": {
      "accountId": "979240842768"
    },
    "offer": {
      "id": "offer-vjjmln0rm2lz686n"
    }
  }
}
response: {
  "FailedEntryCount": 0,
  "Entries": [
    {
     

True

In [ ]:
# Send all event types (one of each)
print("🚀 Sending all event types...")
all_events = generate_and_send_all_events()
print(f"\n📊 Summary: Successfully sent {len(all_events)} out of {len(EVENT_GENERATORS)} event types")

In [ ]:
# Generate multiple events of random types
def generate_random_events(count: int = 10):
    """Generate and send random events"""
    print(f"🎲 Generating {count} random events...")
    sent_count = 0
    
    for i in range(count):
        event_type = random.choice(list(EVENT_GENERATORS.keys()))
        print(f"\n{i+1}/{count}: {event_type}")
        
        event = generate_and_send_event(event_type)
        if event:
            sent_count += 1
        
        # Random delay between 1-3 seconds
        time.sleep(random.uniform(1, 3))
    
    print(f"\n📊 Random events summary: {sent_count}/{count} sent successfully")

# Uncomment to run random event generation
# generate_random_events(5)

In [28]:
event_lic_upd_manu = {
  "version": "0",
  "id": "f38bc7b3-dd36-4107-afae-cbf2e41b598c",
  "detail-type": "License Updated - Manufacturer",
  "source": "mock.agreement-marketplace",
  "account": "305142167625",
  "time": datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
  "region": "us-east-1",
  "resources": [
    "arn:aws:aws-marketplace::aws:agreement:agmt-j31sgfmgq88f8y85kqpsq0j9r"
  ],
  "detail": {
    "requestId": "b678e33a-c297-4f2c-a0cc-4e2e0abf03b0",
    "catalog": "AWSMarketplace",
    "agreementId": "agmt-4w8d12fp5wysfx73frt1phjvt",
    "agreement": {
      "id": "agmt-4w8d12fp5wysfx73frt1phjvt"
    },
    "product": {
      "code": "cqj79vcg9mig83heufcyjpfai",
      "id": "prod-2zch4ci4slnp2"
    },
    "license": {
      "id": "l-cliylbbgx0armw2bd83kqj12fyhatc6y"
    },
    "acceptor": {
      "accountId": "305142167625"
    },
    "proposer": {
      "accountId": "305142167625"
    },
    "offer": {
      "id": "offer-vjjmln0rm2lz686n"
    }
  }
}
send_event_to_eventbridge(event_lic_upd_manu)

response: {
  "FailedEntryCount": 0,
  "Entries": [
    {
      "EventId": "a5b4fa92-2c11-b1a4-35b1-82be4c79d6d3"
    }
  ],
  "ResponseMetadata": {
    "RequestId": "26aa905d-e504-4130-aede-bb59f2040b70",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "26aa905d-e504-4130-aede-bb59f2040b70",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "85",
      "date": "Thu, 11 Sep 2025 15:18:52 GMT"
    },
    "RetryAttempts": 0
  }
}
✅ Successfully sent: License Updated - Manufacturer


True

response: {
  "FailedEntryCount": 0,
  "Entries": [
    {
      "EventId": "d8b8dfde-c89a-a74d-687c-d7347a4f1e41"
    }
  ],
  "ResponseMetadata": {
    "RequestId": "55f6c8b5-34cb-4bd0-ad32-4e504b5b11ae",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "55f6c8b5-34cb-4bd0-ad32-4e504b5b11ae",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "85",
      "date": "Thu, 11 Sep 2025 11:27:04 GMT"
    },
    "RetryAttempts": 0
  }
}
✅ Successfully sent: License Updated - Manufacturer


True

In [26]:
#source = "aws.agreement-marketplace"
source = "mock.agreement-marketplace"
event_lic_deprov_manu = {
  "version": "0",
  "id": "12345678-1234-1234-1234-123456789012",
  "detail-type": "License Deprovisioned - Manufacturer",
  "source": source,
  "account": "305142167625",
  "time": datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
  "region": "us-east-1",
  "resources": [    
       "arn:aws:aws-marketplace::aws:agreement:agmt-4mwg1nevbokzw95eca5797ixs" 
  ],
  "detail": {
    "requestId": "3d4c9f9b-b809-4f5e-9fac-a9ae98b05cbb",
    "catalog": "AWSMarketplace",
     "agreement": {
         "id": "agmt-4mwg1nevbokzw95eca5797ixs",
     },
     "product": {
        "code": "cqj79vcg9mig83heufcyjpfai",
        "id"  : "prod-2zch4ci4slnp2"
     },
     "license": {
       "id": "l-cliylbbgx0armw2bd83kqj12fyhatc6y"
     },
     "acceptor": {
      "accountId": "305142167625"
     },
     "proposer":{
        "accountId": "305142167625"
    },
    "offer": {
      "id": "offer-vjjmln0rm2lz686n"
     }
  }
}
send_event_to_eventbridge(event_lic_deprov_manu)

response: {
  "FailedEntryCount": 0,
  "Entries": [
    {
      "EventId": "f0da2a45-c6c5-c946-b170-9e279e7f7ac5"
    }
  ],
  "ResponseMetadata": {
    "RequestId": "d37e5057-36a1-42f5-8d57-687fbbcd4045",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "d37e5057-36a1-42f5-8d57-687fbbcd4045",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "85",
      "date": "Thu, 11 Sep 2025 14:52:14 GMT"
    },
    "RetryAttempts": 0
  }
}
✅ Successfully sent: License Deprovisioned - Manufacturer


True

## Verification and Monitoring

In [ ]:
# Check SQS queue for received messages (optional - requires queue URL)
def check_sqs_queue(queue_url: str = None):
    """Check SQS queue for received messages"""
    if not queue_url:
        print("⚠️  Please provide SQS queue URL to check messages")
        return
    
    try:
        sqs_client = session.client('sqs')
        response = sqs_client.get_queue_attributes(
            QueueUrl=queue_url,
            AttributeNames=['ApproximateNumberOfMessages', 'ApproximateNumberOfMessagesNotVisible']
        )
        
        visible = response['Attributes']['ApproximateNumberOfMessages']
        not_visible = response['Attributes']['ApproximateNumberOfMessagesNotVisible']
        
        print(f"📊 SQS Queue Status:")
        print(f"  - Visible messages: {visible}")
        print(f"  - Messages being processed: {not_visible}")
        
    except Exception as e:
        print(f"❌ Error checking SQS queue: {str(e)}")

# Example usage (uncomment and provide your queue URL)
# check_sqs_queue('https://sqs.us-east-1.amazonaws.com/YOUR-ACCOUNT/marketplace-events-dev')

In [17]:
# Query DynamoDB table for stored events (optional - requires table name)
def check_dynamodb_events(table_name: str = None, limit: int = 10):
    """Check DynamoDB table for stored events"""
    if not table_name:
        print("⚠️  Please provide DynamoDB table name to check events")
        return
    
    try:
        dynamodb = session.resource('dynamodb')
        table = dynamodb.Table(table_name)
        
        response = table.scan(Limit=limit)
        events = response['Items']
        
        print(f"📊 DynamoDB Events (last {len(events)} events):")
        print(json.dumps(events, indent=2, default=str))
        #for event in events:
        #    print(f"  - {event.get('eventType', 'Unknown')} | {event.get('timestamp', 'No timestamp')} | Agreement: {event.get('agreementId', 'Unknown')}")
            
    except Exception as e:
        print(f"❌ Error checking DynamoDB: {str(e)}")

# Example usage (uncomment and provide your table name)
# check_dynamodb_events('marketplace-events-dev')

In [ ]:
check_dynamodb_events('marketplace-events-dev')

In [18]:
check_dynamodb_events('AWSMarketplaceSubscribersIdentifier')

📊 DynamoDB Events (last 10 events):
[
  {
    "subscription_expired": true,
    "successfully_subscribed": true,
    "entitlement": "{}",
    "customerIdentifier": "l-mxjjojeogfuau61256pcwf01yokidqq1"
  },
  {
    "subscription_expired": true,
    "successfully_subscribed": true,
    "entitlement": "{}",
    "customerIdentifier": "l-1x87zrrnwle2a9l4cnkvdzax31ezsrht"
  },
  {
    "subscription_expired": true,
    "successfully_subscribed": true,
    "entitlement": "{}",
    "customerIdentifier": "l-ftij0qmqgc8zioz6zwvpg37jpwa5e6g7"
  },
  {
    "contactEmail": "psacha@amazon.de",
    "subscription_action": "subscribe-success",
    "successfully_subscribed": true,
    "entitlement": "{\"$metadata\":{\"httpStatusCode\":200,\"requestId\":\"176b1ce0-9cc5-4cdc-8c6f-4d9c2e70d73e\",\"attempts\":1,\"totalRetryDelay\":0},\"Entitlements\":[{\"CustomerAWSAccountId\":\"800176916635\",\"CustomerIdentifier\":\"U2vkwkNg3SZ\",\"Dimension\":\"dimension_1_id\",\"ExpirationDate\":\"2025-10-02T14:30:34.976

## Usage Instructions

1. **Setup**: Run the first few cells to import libraries and set up generators
2. **Preview**: Use the sample event preview to see what events look like
3. **Single Test**: Send one event to test your infrastructure
4. **Full Test**: Send all event types to test comprehensive coverage
5. **Random Load**: Generate random events to simulate realistic load
6. **Monitor**: Check SQS and DynamoDB to verify events are being captured

### Event Types Generated:
- **Purchase Agreement Created**: Manufacturer, Proposer, Acceptor
- **Purchase Agreement Amended**: Manufacturer, Proposer, Acceptor  
- **Purchase Agreement Ended**: Manufacturer, Proposer, Acceptor
- **License Updated**: Manufacturer only
- **License Deprovisioned**: Manufacturer only

### Key Features:
- Realistic test data using Faker library
- Proper event structure matching AWS Marketplace schema
- Channel Partner vs Direct offer scenarios
- Configurable batch sending with delays
- Error handling and success tracking
- Monitoring utilities for verification